#  ⭐ `dim_soc_llt`

In [62]:
%run ../../config/bootstrap.py

import pandas as pd
from pathlib import Path
from utils import get_project_root

In [63]:
project_root = get_project_root()

# 🥉 Bronze 

Raw Data
as is production, low quality

In [64]:
base = (
    project_root
    / "data/01_bronze/meddra/MedDRA_28_1_Brazilian_Portuguese/ascii-281"
)

In [65]:
def read_meddra_ascii(file_path):
    df = pd.read_csv(
        file_path,
        sep=r"\$",
        header=None,
        dtype=str,
        encoding="latin-1",
        engine="python"
    )
    df = df.dropna(axis=1, how="all")
    return df

# 🥈 Silver

normalized data, medium quality

In [66]:
mdhier = read_meddra_ascii(base / "mdhier.asc")

mdhier.columns = [
    "PT_CODE",
    "HLT_CODE",
    "HLGT_CODE",
    "SOC_CODE",
    "PT_NAME",
    "HLT_NAME",
    "HLGT_NAME",
    "SOC_NAME",
    "SOC_ABBREV",
    "PRIMARY_SOC",
    "PRIMARY_FLAG"
]

mdhier.head()

,PT_CODE,HLT_CODE,HLGT_CODE,SOC_CODE,PT_NAME,HLT_NAME,HLGT_NAME,SOC_NAME,SOC_ABBREV,PRIMARY_SOC,PRIMARY_FLAG
0,10002043,10002042,10002086,10005329,Anemia por deficiência de folato,Anemiais carenciais,Anemias não hemolíticas e depressão medular,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,Y
1,10002080,10002042,10002086,10005329,Anemia por carência de vitamina B12,Anemiais carenciais,Anemias não hemolíticas e depressão medular,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,Y
2,10002081,10002042,10002086,10005329,Anemia por carência de vitamina B6,Anemiais carenciais,Anemias não hemolíticas e depressão medular,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,Y
3,10022972,10002042,10002086,10005329,Anemia ferropriva,Anemiais carenciais,Anemias não hemolíticas e depressão medular,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,Y
4,10034695,10002042,10002086,10005329,Anemia perniciosa,Anemiais carenciais,Anemias não hemolíticas e depressão medular,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,Y


In [67]:
llt = read_meddra_ascii(base / "llt.asc")
llt.columns = ["LLT_CODE", "LLT_NAME", "PT_CODE", "LLT_CURRENCY"]

hierarquia_completa = mdhier.merge(
    llt[["LLT_CODE", "LLT_NAME", "PT_CODE"]],
    on="PT_CODE",
    how="left"
)

hierarquia_completa.head()

,PT_CODE,HLT_CODE,HLGT_CODE,SOC_CODE,PT_NAME,HLT_NAME,HLGT_NAME,SOC_NAME,SOC_ABBREV,PRIMARY_SOC,PRIMARY_FLAG,LLT_CODE,LLT_NAME
0,10002043,10002042,10002086,10005329,Anemia por deficiência de folato,Anemiais carenciais,Anemias não hemolíticas e depressão medular,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,Y,10002043,Anemia por deficiência de folato
1,10002043,10002042,10002086,10005329,Anemia por deficiência de folato,Anemiais carenciais,Anemias não hemolíticas e depressão medular,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,Y,10002044,Anemia por deficiência de ácido fólico
2,10002043,10002042,10002086,10005329,Anemia por deficiência de folato,Anemiais carenciais,Anemias não hemolíticas e depressão medular,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,Y,10002281,Anemia por carência de folato
3,10002043,10002042,10002086,10005329,Anemia por deficiência de folato,Anemiais carenciais,Anemias não hemolíticas e depressão medular,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,Y,10002282,Anemia por carência de ácido fólico
4,10002043,10002042,10002086,10005329,Anemia por deficiência de folato,Anemiais carenciais,Anemias não hemolíticas e depressão medular,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,Y,10016881,Anemia por carência de folato


# 🥇 Gold

In [68]:
gold_path = (
    project_root
    / "data/03_gold/dim_soc_llt"
)

gold_path.mkdir(parents=True, exist_ok=True)

arquivo_saida = gold_path / "dim_soc_llt.parquet"

In [69]:
hierarquia_completa.to_parquet(
    arquivo_saida,
    index=False,
    engine="pyarrow"
)

print("Arquivo salvo em:", arquivo_saida)

Arquivo salvo em: C:\Users\janet\Documents\VigiMed\vigimed\data\03_gold\dim_soc_llt\dim_soc_llt.parquet
